# Demo: Catalog searches with CQL2 requests

Demo of work done as part of tickets RSPY-160 and RSPY-656.   
This shows the usage of advanced temporal filters following these specifications: https://pforge-exchange2.astrium.eads.net/confluence/display/COPRS/4.+External+data+selection+policies

## 0 - Initialization

In [1]:
import requests
import os
import pprint
import time
import pystac
from pystac import Asset, Collection, Extent, Item, SpatialExtent, TemporalExtent, ItemCollection
# Init environment before running a demo notebook.
from resources.utils import *

pp = pprint.PrettyPrinter(indent=2, width=80, sort_dicts=False, compact=True)
session = requests.Session()
auxip_client, cadip_client, catalog_client, staging_client, *_ = init_demo()

if os.getenv("RSPY_LOCAL_MODE") == "1":
    href_cadip = "http://rs-server-cadip:8000"
    href_adgs = "http://rs-server-adgs:8000"
else:
    href_cadip = href_adgs = os.environ["RSPY_WEBSITE"]
    session.cookies.set("session", os.environ["RSPY_OAUTH2_COOKIE"])

cadip_collection_id = "cadip_sentinel1"
adgs_collection_id = "adgs"
TIMEOUT = 10
collection_description = Collection(
    id=TEST_COLLECTION,
    description=None,  # rs-client will provide a default description for us
    extent=Extent(
        spatial=SpatialExtent(bboxes=[-180.0, -90.0, 180.0, 90.0]),
        temporal=TemporalExtent([start_date, stop_date]),
    ),
)

# Init the dask cluster
from resources.dask_clusters.dask_main_env import *
await init_dask_cluster_staging()

Auxip service: http://rs-server-adgs:8000/auxip
PRIP service: http://rs-server-prip:8000/prip
CADIP service: http://rs-server-cadip:8000/cadip
Catalog service: http://rs-server-catalog:8000
Staging service: http://rs-server-staging:8000
DPR service: http://nginx:80
OSAM service: http://rs-server-osam:8000


### Run notebook: [notebooks/init-dask-clusters/init_dask_cluster_staging.ipynb](../../init-dask-clusters/init_dask_cluster_staging.ipynb)

Fetching values from processing-storage-configuration prefect variable...
[staging] Command line: 'papermill /home/jovyan/notebooks/init-dask-clusters/init_dask_cluster_staging.ipynb /home/jovyan/.papermill.ipynb --log-output'
[staging] Input Notebook:  /home/jovyan/notebooks/init-dask-clusters/init_dask_cluster_staging.ipynb
[staging] Output Notebook: /home/jovyan/.papermill.ipynb
[staging] [IPKernelApp] WARNING | Kernel is running over TCP without encryption. All communication (including code and outputs) is sent in plain text and is susceptible to eavesdropping. Use IPC transport or launch with kernel manager-provisioned CurveZMQ keys to enable transport encryption.
[staging] Executing notebook with kernel: py3.13.12-2026.7.0
[staging] Executing Cell 1---------------------------------------
[staging] Ending Cell 1------------------------------------------
[staging] Executing Cell 2---------------------------------------
[staging] Python version: 3.13.12
[staging] Dask version: 2026.

ClusterInfo(jupyter_token='***', dask_gateway_address='http://dask-staging:8000', cluster_label='dask-staging', cluster_instance='54bd135f965e469eaf8539ca610cb7d4')

## 1 - Building catalog

The following code is taken from demo called "404_582_rsclient.ipynb".   
This section creates a catalog by staging data from AUXIP and CADIP stations.

In [3]:
# Create a test collection 
collection = create_test_collection()

# Get all the items from the collection "cadip_sentinel1" found in the configuration of the CADIP station
items_collection_cadip = list(cadip_client.get_items(cadip_collection_id))
assert len(items_collection_cadip) > 0

# Request 14 items from the collection "adgs" found in the configuration of the ADGS station
items_collection_adgs = auxip_client.search(max_items = 14, collections = [adgs_collection_id])
assert len(items_collection_adgs) == 14

# Starting 2 staging processes, one from the CADIP station and one from the ADGS station
staging_resp_list = []
for items in [pystac.ItemCollection(list(items_collection_cadip)), pystac.ItemCollection(list(items_collection_adgs))]:
    staging_resp_list.append(staging_client.run_staging(items.to_dict(), TEST_COLLECTION))
    
for resp in staging_resp_list:
    staging_client.wait_for_jobs(resp, logger)

07:52:32.714 [INFO] (rs_client.rs_client) Retrieving all items from collection 'cadip_sentinel1'.
07:52:32.819 [INFO] (rs_client.rs_client) 🔍 Performing STAC search with parameters: {'max_items': 14, 'collections': ['adgs'], 'datetime': None, 'filter': None}
07:52:32.869 [INFO] (rs_client.rs_client) STAC search raw response: <pystac.item_collection.ItemCollection object at 0x7657e34ab230>
07:52:33.524 [INFO] (resources.utils) job_status: {'status': 'running', 'type': 'process', 'created': '2026-07-29T07:52:33Z', 'message': 'In progress', 'processID': 'staging', 'progress': 8, 'started': '2026-07-29T07:52:33Z', 'updated': '2026-07-29T07:52:33Z', 'jobID': 'bac5139b-a497-44ea-a90f-cfec1790c0d4'}
07:52:33.525 [INFO] (resources.utils) ----- Staging from 'cadip-station' job 'bac5139b-a497-44ea-a90f-cfec1790c0d4': RUNNING 

07:52:35.534 [INFO] (resources.utils) job_status: {'status': 'running', 'type': 'process', 'created': '2026-07-29T07:52:33Z', 'message': 'In progress', 'processID': 'stagi

RuntimeError: Staging from 'adgs-station' job 'e4e6f686-5c67-47bb-b7ad-27db3cc409a9': FAILED

In [ ]:
# Check the catalog for my_test_collection
result = list(catalog_client.get_items(TEST_COLLECTION))

for item in result:
    print(f"Item {item.id} has {len(item.assets)} assets")

## 2 - Run various search requests with different filters to retrieve parts of the data

Filters are the ones described here: https://pforge-exchange2.astrium.eads.net/confluence/display/COPRS/4.+External+data+selection+policies

Forked version of Pygeofilter: https://github.com/RS-PYTHON/pygeofilter

In [ ]:
# ValCover filter
# This mode gets all files that cover entirely time interval  [t0 – dt0, t1 + dt1].

valcover_filter =  {
    "op": "t_contains",
    "args": [
        {"interval": [{"property": "start_datetime"}, {"property": "end_datetime"}]},
        {"interval": ["2024-05-27T09:44:12.509000Z", "2024-05-27T09:44:13.509000Z"]}
    ]
}

# http://localhost:8003/catalog/search?collections=ecombelles_my_test_collection&filter=T_CONTAINS(INTERVAL(start_datetime,end_datetime),INTERVAL(TIMESTAMP(%272024-05-27T09:44:12.509000Z%27),TIMESTAMP(%272024-05-27T09:44:13.509000Z%27)))

params = {
    "owner_id": OWNER_ID,
    "max_items": 100,
    "collections": ["my_test_collection"],
    "stac_filter": valcover_filter
}

catalog_client.search(**params)

In [ ]:
# LatestValCover filter
# This mode gets the latest file that covers entirely time interval  [t0 – dt0, t1 + dt1]. The latest record is the one with the more recent Generation Date.

latestvalcover_filter =  {
    "op": "t_contains",
    "args": [
        {"interval": [{"property": "start_datetime"}, {"property": "end_datetime"}]},
        {"interval": ["2024-05-27T09:44:12.509000Z", "2024-05-27T09:44:13.509000Z"]}
    ]
}

# http://localhost:8003/catalog/search?collections=ecombelles_my_test_collection&filter=T_CONTAINS(INTERVAL(start_datetime,end_datetime),INTERVAL(TIMESTAMP(%272024-05-27T09:44:12.509000Z%27),TIMESTAMP(%272024-05-27T09:44:13.509000Z%27)))&sortby=-properties.created&limit=1

params = {
    "owner_id": OWNER_ID,
    "collections": ["my_test_collection"],
    "stac_filter": latestvalcover_filter,
    "sortby": [
        {
            "field": "created",
            "direction": "desc"
        }
    ],
    "max_items": 1,
}

catalog_client.search(**params)

In [ ]:
# ValIntersect filter
# This mode gets all files that cover partly time interval  [t0 – dt0, t1 + dt1].

valintersect_filter =  {
    "op": "t_intersects",
    "args": [
        {"interval": [{"property": "start_datetime"}, {"property": "end_datetime"}]},
        {"interval": ["2024-01-21T09:44:12.509000Z", "2024-06-26T09:44:13.509000Z"]}
    ]
}

# http://localhost:8003/catalog/search?collections=ecombelles_my_test_collection&filter=T_INTERSECTS(INTERVAL(start_datetime,end_datetime),INTERVAL(TIMESTAMP(%272024-01-21T09:44:12.509000Z%27),TIMESTAMP(%272024-06-26T09:44:13.509000Z%27)))

params = {
    "owner_id": OWNER_ID,
    "max_items": 100,
    "collections": ["my_test_collection"],
    "stac_filter": valintersect_filter
}

catalog_client.search(**params)

In [ ]:
# LatestValIntersect filter
# This mode gets the latest file that covers partly time interval  [t0 – dt0 , t1 + dt1]. The latest record is the one with the more recent Generation Date.

latestvalintersect_filter =  {
    "op": "t_intersects",
    "args": [
        {"interval": [{"property": "start_datetime"}, {"property": "end_datetime"}]},
        {"interval": ["2024-01-21T09:44:12.509000Z", "2024-06-26T09:44:13.509000Z"]}
    ]
}

# http://localhost:8003/catalog/search?collections=ecombelles_my_test_collection&filter=T_INTERSECTS(INTERVAL(start_datetime,end_datetime),INTERVAL(TIMESTAMP(%272024-01-21T09:44:12.509000Z%27),TIMESTAMP(%272024-06-26T09:44:13.509000Z%27)))&sortby=-properties.created&limit=1

params = {
    "owner_id": OWNER_ID,
    "collections": ["my_test_collection"],
    "stac_filter": latestvalintersect_filter,
    "sortby": [{"field": "created", "direction": "desc"}],
    "max_items": 1,
}

catalog_client.search(**params)

In [ ]:
# LatestValidity filter
# This mode gets a product with the latest Validity Start Time.

# http://localhost:8003/catalog/search?collections=ecombelles_my_test_collection&sortby=-properties.created&limit=1

params = {
    "owner_id": OWNER_ID,
    "collections": ["my_test_collection"],
    "sortby": [{"field": "created", "direction": "desc"}],
    "max_items": 1,
}

catalog_client.search(**params)

## 3 - Delete the catalog collection

In [ ]:
result = catalog_client.remove_collection(TEST_COLLECTION)
assert result.json()["deleted collection"] == TEST_COLLECTION
pp.pprint(result.json())